# Energy Demand Forecasting with SARIMA

**Dataset:** Open Power System Data — Germany Daily Energy Consumption (2006–2017)  
**Goal:** Forecast daily electricity consumption using a Seasonal ARIMA (SARIMA) model  

## Workflow
1. Load & explore data
2. Stationarity analysis (ADF test, rolling statistics)
3. Transform to stationary series (differencing)
4. ACF / PACF analysis
5. Model selection via `auto_arima`
6. Fit SARIMA and diagnose residuals
7. Forecast with confidence intervals
8. Evaluate (RMSE, MAE, MAPE)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

import pmdarima as pm
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Data Loading & Initial Exploration

In [ ]:
url = 'https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv'
df = pd.read_csv(url, parse_dates=['Date'], index_col='Date')

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

# Forward-fill the handful of missing consumption days
df['Consumption'] = df['Consumption'].ffill()

consumption = df['Consumption'].copy()
print(f'\nConsumption stats:\n{consumption.describe().round(2)}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(consumption, color='steelblue', linewidth=0.8, label='Daily Consumption')
axes[0].set_title('Germany Daily Electricity Consumption (GWh) — 2006 to 2017', fontsize=13)
axes[0].set_ylabel('GWh')
axes[0].legend()

# Monthly resampled for trend clarity
monthly = consumption.resample('M').mean()
axes[1].plot(monthly, color='tomato', linewidth=1.5, label='Monthly Mean')
axes[1].set_title('Monthly Average Consumption', fontsize=13)
axes[1].set_ylabel('GWh')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly seasonality
df['Month'] = df.index.month
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df.boxplot(column='Consumption', by='Month', ax=axes[0])
axes[0].set_xticklabels(month_labels)
axes[0].set_title('Consumption by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('GWh')

# Day-of-week seasonality
df['DayOfWeek'] = df.index.dayofweek
day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
df.boxplot(column='Consumption', by='DayOfWeek', ax=axes[1])
axes[1].set_xticklabels(day_labels)
axes[1].set_title('Consumption by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('GWh')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 2. Stationarity Analysis

ARIMA requires a **stationary** series — one where the mean, variance, and autocorrelation structure are constant over time.  
We use two complementary tests:
- **ADF (Augmented Dickey-Fuller):** H₀ = unit root (non-stationary). Reject H₀ → stationary.
- **KPSS:** H₀ = stationary. Reject H₀ → non-stationary.

In [ ]:
def plot_rolling_stats(series, window=30, title=''):
    rolling_mean = series.rolling(window=window).mean()
    rolling_std  = series.rolling(window=window).std()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(series,       color='steelblue', linewidth=0.7, label='Original', alpha=0.8)
    ax.plot(rolling_mean, color='tomato',    linewidth=2,   label=f'{window}-day Rolling Mean')
    ax.plot(rolling_std,  color='green',     linewidth=2,   label=f'{window}-day Rolling Std', linestyle='--')
    ax.set_title(title or 'Rolling Mean & Standard Deviation', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_rolling_stats(consumption, window=90, title='Original Series — Rolling Statistics (90-day window)')

In [ ]:
def run_stationarity_tests(series, label='Series'):
    print(f'=== Stationarity Tests: {label} ===')

    # ADF
    adf_result = adfuller(series.dropna(), autolag='AIC')
    print(f'\nADF Test:')
    print(f'  Statistic : {adf_result[0]:.4f}')
    print(f'  p-value   : {adf_result[1]:.4f}')
    print(f'  Critical values: ', {k: f'{v:.4f}' for k, v in adf_result[4].items()})
    print(f'  → {"STATIONARY (reject H0)" if adf_result[1] < 0.05 else "NON-STATIONARY (fail to reject H0)"}')

    # KPSS
    kpss_result = kpss(series.dropna(), regression='c', nlags='auto')
    print(f'\nKPSS Test:')
    print(f'  Statistic : {kpss_result[0]:.4f}')
    print(f'  p-value   : {kpss_result[1]:.4f}')
    print(f'  Critical values: ', kpss_result[3])
    print(f'  → {"NON-STATIONARY (reject H0)" if kpss_result[1] < 0.05 else "STATIONARY (fail to reject H0)"}')
    print()

run_stationarity_tests(consumption, label='Original Consumption')

## 3. Transforming to Stationarity

Energy demand shows **strong weekly seasonality (period=7)**. We apply:
1. **Seasonal differencing** (lag=7) to remove weekly seasonality
2. **First-order differencing** if trend remains

In [ ]:
# Seasonal difference (weekly)
consumption_sdiff = consumption.diff(7).dropna()

# First-order difference on top
consumption_diff = consumption_sdiff.diff(1).dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(consumption,       color='steelblue', linewidth=0.8)
axes[0].set_title('Original Series')
axes[0].set_ylabel('GWh')

axes[1].plot(consumption_sdiff, color='darkorange', linewidth=0.8)
axes[1].set_title('After Seasonal Differencing (lag=7)')
axes[1].set_ylabel('ΔGWh')

axes[2].plot(consumption_diff,  color='seagreen', linewidth=0.8)
axes[2].set_title('After Seasonal + First-Order Differencing')
axes[2].set_ylabel('Δ²GWh')

plt.tight_layout()
plt.show()

run_stationarity_tests(consumption_diff, label='Twice-Differenced Series')

## 4. ACF & PACF Analysis

- **ACF** (Autocorrelation Function): guides the **MA (q)** order
- **PACF** (Partial Autocorrelation Function): guides the **AR (p)** order

Spikes at multiples of 7 confirm weekly seasonality.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_acf(consumption_sdiff,  lags=60, ax=axes[0, 0], title='ACF — Seasonal Diff')
plot_pacf(consumption_sdiff, lags=60, ax=axes[0, 1], title='PACF — Seasonal Diff', method='ywm')
plot_acf(consumption_diff,   lags=60, ax=axes[1, 0], title='ACF — Seasonal + First Diff')
plot_pacf(consumption_diff,  lags=60, ax=axes[1, 1], title='PACF — Seasonal + First Diff', method='ywm')

plt.tight_layout()
plt.show()

## 5. Model Selection with `auto_arima`

`pmdarima.auto_arima` performs a stepwise search over `(p, d, q)(P, D, Q, m)` parameter space, selecting the model with the lowest **AIC** (Akaike Information Criterion).

We set:
- `m=7` for weekly seasonality
- `D=1` (seasonal differencing already identified)
- `d=1` (first-order differencing already identified)

In [ ]:
# Hold out last 90 days as test set
TEST_SIZE = 90

train = consumption.iloc[:-TEST_SIZE]
test  = consumption.iloc[-TEST_SIZE:]

print(f'Train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} days)')
print(f'Test : {test.index.min().date()}  → {test.index.max().date()}  ({len(test)} days)')

In [ ]:
print('Running auto_arima (this may take a few minutes)...')

auto_model = pm.auto_arima(
    train,
    d=1, D=1,
    m=7,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    seasonal=True,
    stepwise=True,
    information_criterion='aic',
    trace=True,
    error_action='ignore',
    suppress_warnings=True
)

print(f'\nBest model: SARIMA{auto_model.order}x{auto_model.seasonal_order}')
print(f'AIC: {auto_model.aic():.2f}')
print(auto_model.summary())

## 6. Fit Final SARIMA Model & Residual Diagnostics

In [ ]:
order          = auto_model.order
seasonal_order = auto_model.seasonal_order

sarima = SARIMAX(
    train,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

result = sarima.fit(disp=False)
print(result.summary())

In [ ]:
fig = result.plot_diagnostics(figsize=(16, 10))
plt.suptitle('SARIMA Residual Diagnostics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Forecasting

We generate forecasts for the 90-day test window and visualise them with 95% confidence intervals.

In [ ]:
forecast_obj = result.get_forecast(steps=TEST_SIZE)
forecast_mean = forecast_obj.predicted_mean
forecast_ci   = forecast_obj.conf_int(alpha=0.05)

# Align index with test set
forecast_mean.index = test.index
forecast_ci.index   = test.index

fig, ax = plt.subplots(figsize=(16, 6))

# Show last 6 months of training data for context
context = train.iloc[-180:]
ax.plot(context,       color='steelblue', linewidth=1,   label='Training Data (last 180 days)')
ax.plot(test,          color='black',     linewidth=1.5, label='Actual (test)')
ax.plot(forecast_mean, color='tomato',    linewidth=1.5, label='SARIMA Forecast', linestyle='--')
ax.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    color='tomato', alpha=0.15, label='95% Confidence Interval'
)

ax.axvline(x=test.index[0], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
ax.set_title(f'SARIMA{order}x{seasonal_order} — 90-Day Energy Demand Forecast', fontsize=13)
ax.set_ylabel('Electricity Consumption (GWh)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 8. Model Evaluation

| Metric | Description |
|--------|-------------|
| **RMSE** | Root Mean Squared Error — penalises large errors |
| **MAE**  | Mean Absolute Error — average absolute deviation |
| **MAPE** | Mean Absolute Percentage Error — scale-free interpretability |

In [ ]:
y_true = test.values
y_pred = forecast_mean.values

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print('='*40)
print('       Model Evaluation — Test Set')
print('='*40)
print(f'  RMSE : {rmse:.2f} GWh')
print(f'  MAE  : {mae:.2f} GWh')
print(f'  MAPE : {mape:.2f}%')
print('='*40)

In [ ]:
residuals = pd.Series(y_true - y_pred, index=test.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(residuals, color='purple', linewidth=1)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_title('Forecast Residuals (Actual − Predicted)')
axes[0].set_ylabel('GWh')

axes[1].hist(residuals, bins=20, color='purple', edgecolor='white', alpha=0.8)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('GWh')

plt.tight_layout()
plt.show()

## 9. Conclusions

### What we did
- Loaded and explored 11 years of Germany's daily electricity consumption
- Confirmed non-stationarity with ADF and KPSS tests
- Removed weekly seasonality and trend via seasonal + first-order differencing
- Used `auto_arima` stepwise search to identify optimal SARIMA parameters
- Fitted SARIMA and validated residuals (white noise, approximately normal)
- Generated a 90-day forecast with 95% confidence intervals
- Evaluated with RMSE, MAE, and MAPE

### Key observations
- Energy consumption exhibits strong weekly seasonality (lower on weekends) and annual seasonality (higher in winter)
- After differencing, the series satisfies stationarity — confirmed by both ADF and KPSS
- SARIMA captures the repeating weekly pattern well; MAPE below 5% is considered strong for energy demand forecasting

### Potential extensions
- Include exogenous variables (temperature, holidays) via **SARIMAX**
- Compare against Prophet, LSTM, or Transformer-based models
- Rolling/expanding window cross-validation for more robust evaluation
- Deploy as a REST API for real-time forecasting